In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CONFIG = json.loads((ROOT / 'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [3]:
TAG = 'ar1_s'

In [4]:
pairs = pd.read_csv(INPUT / 'transitions.csv', dtype={'stage':str})
panel = pd.read_csv(INPUT / 'panel.csv', index_col='dblp_id')
q0 = panel['0'].dropna().to_numpy()
DEST = OUTPUT / TAG
DEST.mkdir(exist_ok=True)

In [5]:
rows = []
for label, start, end in [('0',0,0), ('1-4',1,4), ('5-7',5,7), ('8-20',8,19)]:
    block = pairs.loc[pairs.age.between(start, end)]
    X = np.column_stack([np.ones(len(block)), block.z])
    coef = np.linalg.lstsq(X, block.z_next, rcond=None)[0]
    residual = block.z_next - X @ coef
    rows.append({'stage':label, 'start':start, 'end':end, 'intercept':coef[0], 'beta':coef[1],
                 'sigma':np.sqrt(np.mean(residual**2)), 'n':len(block)})
params = pd.DataFrame(rows)
params.to_csv(DEST / 'parameters.csv', index=False)
pd.DataFrame([{'scale':q0.mean(), 'n':len(q0)}]).to_csv(DEST / 'initialization.csv', index=False)
display(params)

,stage,start,end,intercept,beta,sigma,n
0,0,0,0,0.979817,0.402023,0.936911,1963
1,1-4,1,4,0.848278,0.521572,0.856860,7826
2,5-7,5,7,0.664463,0.570536,0.843930,5140
3,8-20,8,19,0.482056,0.652411,0.847893,12104
